In [91]:
"""Generate presentation figures summarising autapse literature trends."""

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [92]:
# Literature-derived percentages for cortex and hippocampus. TAG selects which
# table is rendered by the plotting cells below.
TAG = 'CORTEX'

cols = ["Layer", "E/I", "neuron-type", "%"]
data = np.array([['5', 'Excitatory', 'PC', 56.4],
        ['2/3', 'Excitatory', 'PC', 11.5],
        ['5', 'Inhibitory', 'FS', 85],
        ['5', 'Inhibitory', 'non-FS', 0],
        ['5', 'Inhibitory', 'FS', 27.9],
        ['5', 'Inhibitory', 'FS', 80],
        ['5', 'Inhibitory', 'FS', 74],
        ['2/3', 'Inhibitory', 'FS', 69.4],
        ['2/3', 'Inhibitory', 'non-FS', 7.1],
        ['5', 'Inhibitory', 'FS', 90]])

data_hippo = np.array([['CA1-3', 'Excitatory', 'PC', 0.],
                       ['Sub', 'Excitatory', 'PC', 51.6],
                       ['CA1-3', 'Excitatory', 'PC', 0.],
                       ['CA1-3', 'Inhibitory', 'basket', 100.],
                       ['CA1-3', 'Inhibitory', 'bistratified', 65.],
                       ['CA1-3', 'Inhibitory', 'axo-axonic', 0.],
                       ['CA1-3', 'Inhibitory', 'basket', 100.],
                       ['CA1-3', 'Inhibitory', 'bistratified', 100.],])

# Convert the selected source array into a tidy frame for seaborn plotting.
if TAG == 'CORTEX':
    data1 = pd.DataFrame(data = data, columns = cols)
else:
    data1 = pd.DataFrame(data_hippo, columns = cols)

data1['%'] = pd.to_numeric(data1['%'])

print(data1.head())


  Layer         E/I neuron-type     %
0     5  Excitatory          PC  56.4
1   2/3  Excitatory          PC  11.5
2     5  Inhibitory          FS  85.0
3     5  Inhibitory      non-FS   0.0
4     5  Inhibitory          FS  27.9


In [93]:
# Plot autapse percentages as horizontal bars, faceted by layer/region.
fig, axes = plt.subplots(2, 1, figsize=(5, 11) )

if TAG == 'CORTEX':
    layers = ['2/3', '5']
    neuron_order = ['PC', 'FS', 'non-FS'] # Consistent ordering for y-axis
else:
    layers = ['CA1-3', 'Sub']
    neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'} # Distinct colors for E vs I

for i, layer in enumerate(layers):
    # Filter to the current layer before seaborn calculates summary/error bars.
    df_layer = data1[data1['Layer'] == layer]
    
    sns.barplot(
        data=df_layer,
        x='%',
        y='neuron-type',
        hue='E/I',
        order=neuron_order,
        hue_order=['Excitatory', 'Inhibitory'],
        ax=axes[i],
        errorbar='sd',
        capsize=0.1,
        palette=custom_palette,
        dodge=False
    )
    
    axes[i].set_title(f'Layer {layer}', fontsize=13, pad=10)
    axes[i].set_xlabel('$\\%$ of Neurons with autapses', fontsize=11, labelpad=8)
    axes[i].set_xlim((0., 100.))
    sns.despine(ax=axes[i], top=True, right=True, left=True, bottom=True)
    
    # Keep a single legend to avoid duplicating labels across stacked facets.
    if i == 0:
        axes[i].set_ylabel('Neuron Type', fontsize=11, labelpad=8)
        axes[i].get_legend().remove()
    else:
        axes[i].set_ylabel('')
        axes[i].legend(loc='upper right')

plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'plots//neuron_type_by_ei_faceted_horizontal_{TAG}.png', dpi=300, bbox_inches='tight')
plt.close()


In [94]:
# Average repeated measurements so each donut chart represents one subtype.
df_mean = data1.groupby(['Layer', 'neuron-type', 'E/I'])['%'].mean().reset_index()

# This exploratory layout reserves four columns so cortex and hippocampus exports
# can share a common figure size.
fig, axes = plt.subplots(2, 4, figsize=(12, 8))

if TAG == 'CORTEX':
    layers = ['2/3', '5']
    neuron_order = ['PC', 'FS', 'non-FS']
else:
    layers = ['CA1-3', 'Sub']
    neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'}
remainder_color = '#e5e5e5' # Light gray for neurons without autapses

for row, layer in enumerate(layers):
    for col, neuron in enumerate(neuron_order):
        ax = axes[row, col]
        
        match = df_mean[(df_mean['Layer'] == layer) & (df_mean['neuron-type'] == neuron)]
        
        if not match.empty:
            with_autapses = match['%'].values[0]
            ei_status = match['E/I'].values[0]
            primary_color = custom_palette[ei_status]
        else:
            with_autapses = 0.0
            primary_color = remainder_color
            
        without_autapses = 100.0 - with_autapses
        
        # Donut slices represent autapse-positive cells and the remaining cells.
        slices = [with_autapses, without_autapses]
        colors = [primary_color, remainder_color]
        
        wedges, texts, autotexts = ax.pie(
            slices, 
            startangle=90, 
            colors=colors,
            autopct=lambda p: f'{p:.1f}%' if p > 0 else '', # Hide label if 0%
            pctdistance=0.55,
            wedgeprops=dict(width=0.4, edgecolor='w') # Donut-style cutout for clean look
        )
        
        for at in autotexts:
            at.set_fontsize(18)
            at.set_weight('bold')
            at.set_color('black')

        ax.set_title(f'Layer {layer}: {neuron}', fontsize=12, pad=8)

plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=15, y=0.98)
plt.tight_layout()
plt.savefig(f'plots//neuron_type_pie_charts_{TAG}.png', dpi=300, bbox_inches='tight')
plt.close()


In [95]:
# Refined donut layout: place the percentage label on the colored autapse slice
# and suppress the gray remainder label.
df_mean = data1.groupby(['Layer', 'neuron-type', 'E/I'])['%'].mean().reset_index()

if TAG == 'CORTEX':
    layers = ['2/3', '5']
    neuron_order = ['PC', 'FS', 'non-FS']
else:
    layers = ['CA1-3', 'Sub']
    neuron_order = ['PC', 'basket', 'bistratified', 'axo-axonic']

fig, axes = plt.subplots(len(layers), len(neuron_order), figsize=(12, 8))

custom_palette = {'Excitatory': '#2678fc', 'Inhibitory': '#f20c4d'}
remainder_color = '#e5e5e5' # Light gray for neurons without autapses

for row, layer in enumerate(layers):
    for col, neuron in enumerate(neuron_order):
        ax = axes[row, col]
        
        match = df_mean[(df_mean['Layer'] == layer) & (df_mean['neuron-type'] == neuron)]
        
        if not match.empty:
            with_autapses = match['%'].values[0]
            ei_status = match['E/I'].values[0]
            primary_color = custom_palette[ei_status]
        else:
            with_autapses = 0.0
            primary_color = remainder_color
            
        without_autapses = 100.0 - with_autapses
        
        slices = [with_autapses, without_autapses]
        colors = [primary_color, remainder_color]
        
        wedges, texts, autotexts = ax.pie(
            slices, 
            startangle=90, 
            colors=colors,
            autopct=lambda p: f'{p:.1f}%',
            pctdistance=0.78,
            wedgeprops=dict(width=0.4, edgecolor='w')
        )
        
        if len(autotexts) > 1:
            autotexts[1].set_text('')
            
        autotexts[0].set_fontsize(13)
        autotexts[0].set_weight('bold')
        autotexts[0].set_color('black')

        ax.set_title(f'Layer {layer}: {neuron}', fontsize=12, pad=8)

plt.suptitle('Proportion ($\\%$) of Neurons Forming Autapses', fontsize=15, y=0.98)
plt.tight_layout()
plt.savefig(f'plots//neuron_type_pie_charts_{TAG}.png', dpi=300, bbox_inches='tight')
plt.close()


In [96]:
# Literature-derived proportions for projection-specific autapse occurrence.
cols = ['connection', 'proportion']
data2 = np.array([['PFC -> Hb', 84.8],
                  ['PFC -> Pons', 59.3],
                  ['PFC -> cPFC', 2.6]])
data_hippo = np.array([['Sub -> NAc', 46.6],
                       ['Sub -> Amygdala', 17.2]])

if TAG == 'CORTEX':
    df2 = pd.DataFrame(data = data2, columns = cols)
    df2['proportion'] = pd.to_numeric(df2['proportion'])
else:
    df2 = pd.DataFrame(data = data_hippo, columns = cols)
    df2['proportion'] = pd.to_numeric(df2['proportion'])
print(df2.head())


    connection  proportion
0    PFC -> Hb        84.8
1  PFC -> Pons        59.3
2  PFC -> cPFC         2.6


In [97]:
# Draw one donut chart per projection, stacked vertically for presentation use.
fig, axes = plt.subplots(df2.shape[0], 1, figsize=(5, 9))

primary_color = '#2678fc'
remainder_color = '#e5e5e5'

for i, row in df2.iterrows():
    ax = axes[i]
    prop = row['proportion']
    conn = row['connection']
    
    slices = [prop, 100 - prop]
    
    wedges, texts = ax.pie(
        slices,
        startangle=90,
        colors=[primary_color, remainder_color],
        wedgeprops=dict(width=0.4, edgecolor='w') 
    )
    
    # Center the percentage label inside the donut hole.
    ax.text(
        0, 0, 
        f'{prop:.1f}%', 
        ha='center', 
        va='center', 
        fontsize=12, 
        fontweight='bold', 
        color='black'
    )
        
    ax.set_ylabel(conn, rotation=0, labelpad=60, va='center', fontsize=12, fontweight='bold')

plt.suptitle('Proportion ($\\%$) by Connection Type', fontsize=14, y=0.98, fontweight='bold')
plt.tight_layout()
plt.savefig(f'plots//connection_pie_charts_{TAG}.png', dpi=300, bbox_inches='tight')
plt.close()
